# 🏦 JPMC Consumer Banking - Hands-on Lab
## Notebook 01: Synthetic Data Generation

---

### What You'll Learn
- Generating realistic consumer banking datasets using Python (Faker + Snowpark)
- Working with semi-structured data (JSON/VARIANT columns)
- Unloading data to internal stages in Parquet format

### Data Model Overview

| Table | Rows | Description |
|-------|------|-------------|
| CUSTOMERS | 100,000 | Customer profiles with demographics |
| ACCOUNTS | 200,000 | Bank accounts (checking, savings, money market) |
| TRANSACTIONS | 500,000 | Financial transactions across channels |
| LOANS | 50,000 | Loan products (mortgage, auto, personal) |
| CREDIT_CARDS | 100,000 | Credit card accounts and balances |
| SUPPORT_TICKETS | 50,000 | Customer support interactions (unstructured text) |

> **Note:** This notebook uses the `HOL_SNOWPARK_OPT_WH` warehouse because we're running Python-heavy DataFrame operations.

In [ ]:
-- Setup context
USE ROLE ACCOUNTADMIN;
USE DATABASE JPMC_CONSUMER_BANKING_HOL;
USE SCHEMA HOL_LAB;
USE WAREHOUSE HOL_SNOWPARK_OPT_WH;

## Step 1: Install Required Libraries and Initialize Session

In [ ]:
# Import packages
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark import functions as F
from snowflake.snowpark.types import *
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random
import json
import hashlib
import uuid

session = get_active_session()
print(f"Connected to: {session.get_current_database()}.{session.get_current_schema()}")
print(f"Warehouse: {session.get_current_warehouse()}")

## Step 2: Generate CUSTOMERS Data (100K rows)

Customer profiles with demographics, segmentation, and risk ratings.

In [ ]:
# =============================================================================
# CUSTOMERS - 100K rows
# =============================================================================

NUM_CUSTOMERS = 100000

# Seed for reproducibility
np.random.seed(42)
random.seed(42)

# Reference data
first_names = ['James', 'Mary', 'Robert', 'Patricia', 'John', 'Jennifer', 'Michael', 'Linda', 
               'David', 'Elizabeth', 'William', 'Barbara', 'Richard', 'Susan', 'Joseph', 'Jessica',
               'Thomas', 'Sarah', 'Christopher', 'Karen', 'Charles', 'Lisa', 'Daniel', 'Nancy',
               'Matthew', 'Betty', 'Anthony', 'Margaret', 'Mark', 'Sandra', 'Donald', 'Ashley',
               'Steven', 'Kimberly', 'Paul', 'Emily', 'Andrew', 'Donna', 'Joshua', 'Michelle']

last_names = ['Smith', 'Johnson', 'Williams', 'Brown', 'Jones', 'Garcia', 'Miller', 'Davis',
              'Rodriguez', 'Martinez', 'Hernandez', 'Lopez', 'Gonzalez', 'Wilson', 'Anderson',
              'Thomas', 'Taylor', 'Moore', 'Jackson', 'Martin', 'Lee', 'Perez', 'Thompson',
              'White', 'Harris', 'Sanchez', 'Clark', 'Ramirez', 'Lewis', 'Robinson']

states = ['NY', 'CA', 'TX', 'FL', 'IL', 'PA', 'OH', 'GA', 'NC', 'MI', 
          'NJ', 'VA', 'WA', 'AZ', 'MA', 'TN', 'IN', 'MO', 'MD', 'WI']

cities_by_state = {
    'NY': ['New York', 'Brooklyn', 'Queens', 'Buffalo', 'Rochester'],
    'CA': ['Los Angeles', 'San Francisco', 'San Diego', 'Sacramento', 'San Jose'],
    'TX': ['Houston', 'Dallas', 'Austin', 'San Antonio', 'Fort Worth'],
    'FL': ['Miami', 'Orlando', 'Tampa', 'Jacksonville', 'Fort Lauderdale'],
    'IL': ['Chicago', 'Aurora', 'Naperville', 'Joliet', 'Rockford'],
    'PA': ['Philadelphia', 'Pittsburgh', 'Allentown', 'Erie', 'Reading'],
    'OH': ['Columbus', 'Cleveland', 'Cincinnati', 'Toledo', 'Akron'],
    'GA': ['Atlanta', 'Augusta', 'Savannah', 'Athens', 'Macon'],
    'NC': ['Charlotte', 'Raleigh', 'Greensboro', 'Durham', 'Winston-Salem'],
    'MI': ['Detroit', 'Grand Rapids', 'Warren', 'Sterling Heights', 'Ann Arbor'],
    'NJ': ['Newark', 'Jersey City', 'Paterson', 'Elizabeth', 'Edison'],
    'VA': ['Virginia Beach', 'Norfolk', 'Chesapeake', 'Richmond', 'Newport News'],
    'WA': ['Seattle', 'Spokane', 'Tacoma', 'Vancouver', 'Bellevue'],
    'AZ': ['Phoenix', 'Tucson', 'Mesa', 'Chandler', 'Scottsdale'],
    'MA': ['Boston', 'Worcester', 'Springfield', 'Cambridge', 'Lowell'],
    'TN': ['Nashville', 'Memphis', 'Knoxville', 'Chattanooga', 'Clarksville'],
    'IN': ['Indianapolis', 'Fort Wayne', 'Evansville', 'South Bend', 'Carmel'],
    'MO': ['Kansas City', 'St. Louis', 'Springfield', 'Columbia', 'Independence'],
    'MD': ['Baltimore', 'Frederick', 'Rockville', 'Gaithersburg', 'Bowie'],
    'WI': ['Milwaukee', 'Madison', 'Green Bay', 'Kenosha', 'Racine']
}

segments = ['Premium', 'Gold', 'Silver', 'Standard']
segment_weights = [0.1, 0.2, 0.3, 0.4]
risk_ratings = ['Low', 'Medium', 'High']
risk_weights = [0.5, 0.35, 0.15]

print("Generating 100K customer records...")

customers_data = []
for i in range(NUM_CUSTOMERS):
    cust_id = f"CUST{i+1:08d}"
    fname = random.choice(first_names)
    lname = random.choice(last_names)
    state = random.choice(states)
    city = random.choice(cities_by_state[state])
    
    # Create address as JSON (semi-structured data)
    address = json.dumps({
        "street": f"{random.randint(100, 9999)} {random.choice(['Main', 'Oak', 'Park', 'Cedar', 'Elm', 'Maple', 'Pine', 'Washington', 'Lake', 'Hill'])} {random.choice(['St', 'Ave', 'Blvd', 'Dr', 'Ln'])}",
        "city": city,
        "state": state,
        "zip": f"{random.randint(10000, 99999)}"
    })
    
    dob = datetime(1950, 1, 1) + timedelta(days=random.randint(0, 25000))
    ssn_hash = hashlib.sha256(f"SSN-{i}".encode()).hexdigest()[:16]
    
    customers_data.append({
        'CUSTOMER_ID': cust_id,
        'FIRST_NAME': fname,
        'LAST_NAME': lname,
        'EMAIL': f"{fname.lower()}.{lname.lower()}{random.randint(1,999)}@email.com",
        'PHONE': f"+1{random.randint(2000000000, 9999999999)}",
        'SSN_HASH': ssn_hash,
        'DATE_OF_BIRTH': dob.strftime('%Y-%m-%d'),
        'ADDRESS_JSON': address,
        'SEGMENT': random.choices(segments, weights=segment_weights, k=1)[0],
        'RISK_RATING': random.choices(risk_ratings, weights=risk_weights, k=1)[0],
        'CREATED_AT': (datetime(2018, 1, 1) + timedelta(days=random.randint(0, 2500))).strftime('%Y-%m-%d %H:%M:%S'),
        'IS_ACTIVE': random.choices([True, False], weights=[0.92, 0.08], k=1)[0]
    })

customers_df = pd.DataFrame(customers_data)
print(f"Generated {len(customers_df)} customer records")
customers_df.head(3)

## Step 3: Generate ACCOUNTS Data (200K rows)

Multiple accounts per customer (checking, savings, money market, CD).

In [ ]:
# =============================================================================
# ACCOUNTS - 200K rows (avg 2 accounts per customer)
# =============================================================================

NUM_ACCOUNTS = 200000

account_types = ['CHECKING', 'SAVINGS', 'MONEY_MARKET', 'CD']
account_type_weights = [0.4, 0.35, 0.15, 0.1]
account_statuses = ['ACTIVE', 'INACTIVE', 'CLOSED', 'FROZEN']
account_status_weights = [0.85, 0.08, 0.05, 0.02]
branches = [f"BRANCH_{i:04d}" for i in range(1, 201)]  # 200 branches

print("Generating 200K account records...")

accounts_data = []
customer_ids = [f"CUST{i+1:08d}" for i in range(NUM_CUSTOMERS)]

for i in range(NUM_ACCOUNTS):
    acct_type = random.choices(account_types, weights=account_type_weights, k=1)[0]
    
    # Balance ranges based on account type
    if acct_type == 'CHECKING':
        balance = round(random.uniform(100, 50000), 2)
    elif acct_type == 'SAVINGS':
        balance = round(random.uniform(500, 150000), 2)
    elif acct_type == 'MONEY_MARKET':
        balance = round(random.uniform(5000, 500000), 2)
    else:  # CD
        balance = round(random.uniform(10000, 250000), 2)
    
    open_date = datetime(2015, 1, 1) + timedelta(days=random.randint(0, 3500))
    
    accounts_data.append({
        'ACCOUNT_ID': f"ACCT{i+1:08d}",
        'CUSTOMER_ID': random.choice(customer_ids),
        'ACCOUNT_TYPE': acct_type,
        'BALANCE': balance,
        'AVAILABLE_BALANCE': round(balance * random.uniform(0.85, 1.0), 2),
        'INTEREST_RATE': round(random.uniform(0.01, 5.25), 2),
        'OPEN_DATE': open_date.strftime('%Y-%m-%d'),
        'STATUS': random.choices(account_statuses, weights=account_status_weights, k=1)[0],
        'BRANCH_ID': random.choice(branches),
        'OVERDRAFT_PROTECTION': random.choices([True, False], weights=[0.6, 0.4], k=1)[0],
        'LAST_ACTIVITY_DATE': (datetime.now() - timedelta(days=random.randint(0, 180))).strftime('%Y-%m-%d')
    })

accounts_df = pd.DataFrame(accounts_data)
print(f"Generated {len(accounts_df)} account records")
accounts_df.head(3)

## Step 4: Generate TRANSACTIONS Data (500K rows)

Financial transactions across multiple channels with merchant categorization.

In [ ]:
# =============================================================================
# TRANSACTIONS - 500K rows
# =============================================================================

NUM_TRANSACTIONS = 500000

txn_types = ['DEBIT', 'CREDIT', 'TRANSFER', 'PAYMENT', 'DEPOSIT', 'WITHDRAWAL', 'FEE']
txn_type_weights = [0.30, 0.20, 0.15, 0.15, 0.10, 0.08, 0.02]

channels = ['ONLINE', 'MOBILE', 'ATM', 'BRANCH', 'POS', 'ACH']
channel_weights = [0.30, 0.25, 0.15, 0.10, 0.15, 0.05]

categories = ['GROCERIES', 'DINING', 'GAS', 'UTILITIES', 'ENTERTAINMENT', 'HEALTHCARE',
              'SHOPPING', 'TRAVEL', 'INSURANCE', 'EDUCATION', 'RENT', 'TRANSFER']
category_weights = [0.15, 0.12, 0.10, 0.10, 0.08, 0.08, 0.12, 0.05, 0.06, 0.04, 0.05, 0.05]

merchants = {
    'GROCERIES': ['Walmart', 'Costco', 'Kroger', 'Whole Foods', 'Trader Joes', 'Safeway'],
    'DINING': ['McDonalds', 'Starbucks', 'Chipotle', 'Olive Garden', 'Panera Bread'],
    'GAS': ['Shell', 'BP', 'Chevron', 'ExxonMobil', 'Sunoco'],
    'UTILITIES': ['ConEdison', 'PG&E', 'Duke Energy', 'Comcast', 'AT&T'],
    'ENTERTAINMENT': ['Netflix', 'Spotify', 'AMC Theaters', 'Disney+', 'HBO Max'],
    'HEALTHCARE': ['CVS Pharmacy', 'Walgreens', 'LabCorp', 'Quest Diagnostics'],
    'SHOPPING': ['Amazon', 'Target', 'Best Buy', 'Home Depot', 'Macys'],
    'TRAVEL': ['United Airlines', 'Hilton Hotels', 'Uber', 'Lyft', 'Expedia'],
    'INSURANCE': ['Geico', 'State Farm', 'Allstate', 'Progressive'],
    'EDUCATION': ['Tuition Payment', 'Student Loan Corp', 'Coursera', 'Pearson'],
    'RENT': ['Property Management LLC', 'Rent Payment', 'Landlord Payment'],
    'TRANSFER': ['Zelle Transfer', 'Venmo', 'Internal Transfer', 'Wire Transfer']
}

txn_statuses = ['COMPLETED', 'PENDING', 'FAILED', 'REVERSED']
txn_status_weights = [0.90, 0.05, 0.03, 0.02]

account_ids = [f"ACCT{i+1:08d}" for i in range(NUM_ACCOUNTS)]

print("Generating 500K transaction records...")

transactions_data = []
for i in range(NUM_TRANSACTIONS):
    category = random.choices(categories, weights=category_weights, k=1)[0]
    txn_type = random.choices(txn_types, weights=txn_type_weights, k=1)[0]
    
    # Amount ranges vary by category
    if category in ['RENT', 'INSURANCE', 'EDUCATION']:
        amount = round(random.uniform(200, 5000), 2)
    elif category == 'TRAVEL':
        amount = round(random.uniform(50, 3000), 2)
    elif category in ['UTILITIES']:
        amount = round(random.uniform(30, 500), 2)
    else:
        amount = round(random.uniform(5, 500), 2)
    
    txn_date = datetime(2024, 1, 1) + timedelta(
        days=random.randint(0, 540),
        hours=random.randint(0, 23),
        minutes=random.randint(0, 59)
    )
    
    transactions_data.append({
        'TXN_ID': f"TXN{i+1:010d}",
        'ACCOUNT_ID': random.choice(account_ids),
        'TXN_DATE': txn_date.strftime('%Y-%m-%d %H:%M:%S'),
        'TXN_TYPE': txn_type,
        'AMOUNT': amount,
        'CURRENCY': 'USD',
        'MERCHANT': random.choice(merchants[category]),
        'CATEGORY': category,
        'CHANNEL': random.choices(channels, weights=channel_weights, k=1)[0],
        'STATUS': random.choices(txn_statuses, weights=txn_status_weights, k=1)[0],
        'REFERENCE_NUM': str(uuid.uuid4())[:12].upper(),
        'DESCRIPTION': f"{txn_type} - {random.choice(merchants[category])} - {category}"
    })

transactions_df = pd.DataFrame(transactions_data)
print(f"Generated {len(transactions_df)} transaction records")
transactions_df.head(3)

## Step 5: Generate LOANS Data (50K rows)

Loan products including mortgages, auto loans, personal loans, and student loans.

In [ ]:
# =============================================================================
# LOANS - 50K rows
# =============================================================================

NUM_LOANS = 50000

loan_types = ['MORTGAGE', 'AUTO', 'PERSONAL', 'STUDENT', 'HOME_EQUITY']
loan_type_weights = [0.35, 0.25, 0.20, 0.12, 0.08]

loan_statuses = ['CURRENT', 'DELINQUENT_30', 'DELINQUENT_60', 'DELINQUENT_90', 'DEFAULT', 'PAID_OFF']
loan_status_weights = [0.70, 0.10, 0.05, 0.03, 0.02, 0.10]

print("Generating 50K loan records...")

loans_data = []
for i in range(NUM_LOANS):
    loan_type = random.choices(loan_types, weights=loan_type_weights, k=1)[0]
    
    # Principal and term based on loan type
    if loan_type == 'MORTGAGE':
        principal = round(random.uniform(150000, 1200000), 2)
        term_months = random.choice([180, 240, 360])
        rate = round(random.uniform(3.0, 7.5), 3)
    elif loan_type == 'AUTO':
        principal = round(random.uniform(15000, 85000), 2)
        term_months = random.choice([36, 48, 60, 72])
        rate = round(random.uniform(3.5, 12.0), 3)
    elif loan_type == 'PERSONAL':
        principal = round(random.uniform(5000, 50000), 2)
        term_months = random.choice([12, 24, 36, 48, 60])
        rate = round(random.uniform(6.0, 24.0), 3)
    elif loan_type == 'STUDENT':
        principal = round(random.uniform(10000, 200000), 2)
        term_months = random.choice([120, 180, 240])
        rate = round(random.uniform(3.0, 8.0), 3)
    else:  # HOME_EQUITY
        principal = round(random.uniform(25000, 300000), 2)
        term_months = random.choice([60, 120, 180])
        rate = round(random.uniform(4.0, 10.0), 3)
    
    origination_date = datetime(2015, 1, 1) + timedelta(days=random.randint(0, 3500))
    monthly_payment = round(principal * (rate/100/12) / (1 - (1 + rate/100/12)**(-term_months)), 2)
    
    loans_data.append({
        'LOAN_ID': f"LOAN{i+1:08d}",
        'CUSTOMER_ID': random.choice(customer_ids),
        'LOAN_TYPE': loan_type,
        'PRINCIPAL_AMOUNT': principal,
        'CURRENT_BALANCE': round(principal * random.uniform(0.3, 0.98), 2),
        'INTEREST_RATE': rate,
        'TERM_MONTHS': term_months,
        'MONTHLY_PAYMENT': monthly_payment,
        'ORIGINATION_DATE': origination_date.strftime('%Y-%m-%d'),
        'MATURITY_DATE': (origination_date + timedelta(days=term_months*30)).strftime('%Y-%m-%d'),
        'STATUS': random.choices(loan_statuses, weights=loan_status_weights, k=1)[0],
        'CREDIT_SCORE_AT_ORIGINATION': random.randint(580, 850),
        'DEBT_TO_INCOME_RATIO': round(random.uniform(0.15, 0.55), 3),
        'COLLATERAL_VALUE': round(principal * random.uniform(1.0, 1.5), 2) if loan_type in ['MORTGAGE', 'AUTO', 'HOME_EQUITY'] else None
    })

loans_df = pd.DataFrame(loans_data)
print(f"Generated {len(loans_df)} loan records")
loans_df.head(3)

## Step 6: Generate CREDIT_CARDS Data (100K rows)

Credit card accounts with utilization, rewards, and payment history.

In [ ]:
# =============================================================================
# CREDIT_CARDS - 100K rows
# =============================================================================

NUM_CREDIT_CARDS = 100000

card_types = ['VISA_PLATINUM', 'VISA_GOLD', 'MASTERCARD_WORLD', 'MASTERCARD_STANDARD', 'AMEX_PREFERRED']
card_type_weights = [0.15, 0.25, 0.20, 0.30, 0.10]

card_statuses = ['ACTIVE', 'SUSPENDED', 'CLOSED', 'LOST_STOLEN']
card_status_weights = [0.88, 0.05, 0.05, 0.02]

print("Generating 100K credit card records...")

credit_cards_data = []
for i in range(NUM_CREDIT_CARDS):
    card_type = random.choices(card_types, weights=card_type_weights, k=1)[0]
    
    # Credit limit based on card type
    if 'PLATINUM' in card_type or 'PREFERRED' in card_type:
        credit_limit = random.choice([15000, 20000, 25000, 30000, 50000, 75000, 100000])
    elif 'GOLD' in card_type or 'WORLD' in card_type:
        credit_limit = random.choice([5000, 7500, 10000, 15000, 20000])
    else:
        credit_limit = random.choice([1000, 2000, 3000, 5000, 7500, 10000])
    
    utilization = random.uniform(0.0, 0.95)
    balance = round(credit_limit * utilization, 2)
    
    issue_date = datetime(2016, 1, 1) + timedelta(days=random.randint(0, 3000))
    
    credit_cards_data.append({
        'CARD_ID': f"CARD{i+1:08d}",
        'CUSTOMER_ID': random.choice(customer_ids),
        'CARD_TYPE': card_type,
        'CARD_NUMBER_HASH': hashlib.sha256(f"CARD-{i}".encode()).hexdigest()[:16],
        'CREDIT_LIMIT': credit_limit,
        'CURRENT_BALANCE': balance,
        'AVAILABLE_CREDIT': round(credit_limit - balance, 2),
        'APR': round(random.uniform(12.99, 29.99), 2),
        'MINIMUM_PAYMENT': round(max(25, balance * 0.02), 2),
        'PAYMENT_DUE_DATE': (datetime.now() + timedelta(days=random.randint(1, 30))).strftime('%Y-%m-%d'),
        'REWARDS_POINTS': random.randint(0, 250000),
        'CASH_BACK_EARNED': round(random.uniform(0, 5000), 2),
        'ISSUE_DATE': issue_date.strftime('%Y-%m-%d'),
        'EXPIRATION_DATE': (issue_date + timedelta(days=random.choice([1095, 1460, 1825]))).strftime('%Y-%m-%d'),
        'STATUS': random.choices(card_statuses, weights=card_status_weights, k=1)[0],
        'AUTOPAY_ENABLED': random.choices([True, False], weights=[0.65, 0.35], k=1)[0]
    })

credit_cards_df = pd.DataFrame(credit_cards_data)
print(f"Generated {len(credit_cards_df)} credit card records")
credit_cards_df.head(3)

## Step 7: Generate SUPPORT_TICKETS Data (50K rows) - Unstructured Text

Customer support interactions with realistic free-text descriptions. This data will be used later for Cortex AI function processing.

In [ ]:
# =============================================================================
# SUPPORT_TICKETS - 50K rows (with unstructured text for Cortex AI)
# =============================================================================

NUM_TICKETS = 50000

ticket_categories = ['BILLING', 'TECHNICAL', 'FRAUD', 'ACCOUNT_CLOSURE', 'GENERAL_INQUIRY', 'DISPUTE']
ticket_cat_weights = [0.25, 0.20, 0.15, 0.10, 0.20, 0.10]

priorities = ['LOW', 'MEDIUM', 'HIGH', 'CRITICAL']
priority_weights = [0.30, 0.40, 0.20, 0.10]

ticket_channels = ['PHONE', 'EMAIL', 'CHAT', 'MOBILE_APP', 'BRANCH', 'SOCIAL_MEDIA']
ticket_channel_weights = [0.25, 0.20, 0.25, 0.15, 0.10, 0.05]

ticket_statuses = ['OPEN', 'IN_PROGRESS', 'RESOLVED', 'CLOSED', 'ESCALATED']
ticket_status_weights = [0.15, 0.20, 0.35, 0.25, 0.05]

# Realistic ticket templates by category
ticket_templates = {
    'BILLING': [
        "I was charged ${amount} on {date} but I didn't authorize this transaction at {merchant}. My account number ends in {acct_last4}. Please investigate and reverse this charge immediately. I've been a loyal customer for {years} years.",
        "My monthly statement shows a fee of ${amount} for overdraft protection that I never signed up for. I'd like this removed from account ending in {acct_last4}. This has been happening for the past {months} months.",
        "I need help understanding why my interest rate changed from {old_rate}% to {new_rate}% without any notification. Account {acct_last4}. I've always made payments on time.",
        "Double charge of ${amount} from {merchant} on {date}. This is the third time this has happened. Please fix this and credit my account {acct_last4} immediately.",
        "I was told my payment of ${amount} was received on {date} but my balance still shows the old amount. Reference number {ref_num}. Please update my account."
    ],
    'TECHNICAL': [
        "I cannot log into my online banking since {date}. I've tried resetting my password multiple times but keep getting error code E-{error_code}. My username is associated with account ending in {acct_last4}.",
        "The mobile app crashes every time I try to view my transaction history. I'm using iPhone {phone_model} with iOS {ios_version}. This started after the latest update.",
        "I'm unable to complete wire transfers through the website. Getting timeout errors after entering the recipient information. Transfer amount of ${amount} to account {dest_acct}.",
        "My Zelle payments are not going through. I've been trying to send ${amount} to {recipient} since {date}. Error message says 'service temporarily unavailable'.",
        "Two-factor authentication is not working. I'm not receiving the SMS codes to my phone number ending in {phone_last4}. I need to access my account urgently for a ${amount} payment due today."
    ],
    'FRAUD': [
        "URGENT: I see {num_txns} transactions I didn't make totaling ${amount} on my account ending in {acct_last4}. Transactions from {merchant} in {location}. I was in {home_city} at the time. Please freeze my account immediately!",
        "Someone opened a credit card in my name. I received a statement for card ending in {card_last4} with a balance of ${amount}. I never applied for this card. This is identity theft.",
        "I received a notification about a ${amount} wire transfer from my account {acct_last4} to an unknown recipient. I did NOT authorize this! Please stop this transfer immediately.",
        "Multiple ATM withdrawals of ${amount} each from locations I've never been to. Total unauthorized withdrawals: ${total}. My card was in my possession the entire time. Account {acct_last4}.",
        "I got an email claiming to be from Chase asking me to verify my account. I clicked the link and entered my credentials before realizing it was fake. Please secure my account {acct_last4} immediately."
    ],
    'ACCOUNT_CLOSURE': [
        "I'd like to close my {account_type} account ending in {acct_last4}. Current balance is ${amount}. Please transfer remaining funds to my other account ending in {dest_acct} and send confirmation.",
        "Requesting closure of all accounts under my name. I'm switching to {competitor_bank}. Please provide final statements and confirm there are no pending transactions.",
        "My mother passed away on {date} and I need to close her accounts. Her name is {name} with account ending in {acct_last4}. I have the death certificate and am the executor of her estate.",
        "I want to close my credit card ending in {card_last4} due to the annual fee increase. Balance is ${amount}. Please confirm no further charges will be applied.",
        "Closing my joint account ending in {acct_last4} due to divorce. My attorney {attorney_name} will be in contact regarding the fund distribution."
    ],
    'GENERAL_INQUIRY': [
        "What's the current interest rate for a {loan_type} loan? My credit score is approximately {credit_score} and I'm looking to borrow ${amount} over {term} years.",
        "I'd like information about your Premium checking account. What are the requirements to waive the monthly fee? My current balance is typically around ${amount}.",
        "Can you explain the difference between your money market and high-yield savings accounts? I have ${amount} to deposit and want the best return.",
        "I'm planning to buy a house for approximately ${amount}. What documents do I need to start the mortgage pre-approval process? My annual income is ${income}.",
        "How do I set up automatic transfers from my checking to savings account? I'd like to transfer ${amount} on the {day}th of every month."
    ],
    'DISPUTE': [
        "I'm disputing a charge of ${amount} from {merchant} on {date}. I returned the merchandise on {return_date} but never received my refund. I have the return receipt (tracking #{tracking}).",
        "The service I paid ${amount} for on {date} was never provided by {merchant}. I've contacted them {num_times} times with no resolution. Account ending in {acct_last4}.",
        "I was charged ${amount} for a subscription I cancelled on {cancel_date}. {merchant} continues to charge me monthly. This is the {month_num}th unauthorized charge.",
        "The amount charged was ${amount} but the receipt shows ${correct_amount}. This happened at {merchant} on {date}. Difference of ${difference} needs to be refunded to account {acct_last4}.",
        "I paid ${amount} via wire transfer on {date} for services from {merchant} (ref: {ref_num}). The company is unresponsive and I believe this is a scam. Please help recover my funds."
    ]
}

print("Generating 50K support ticket records with realistic text...")

support_tickets_data = []
for i in range(NUM_TICKETS):
    category = random.choices(ticket_categories, weights=ticket_cat_weights, k=1)[0]
    template = random.choice(ticket_templates[category])
    
    # Fill template placeholders
    description = template.format(
        amount=f"{random.uniform(10, 15000):.2f}",
        date=(datetime.now() - timedelta(days=random.randint(1, 60))).strftime('%m/%d/%Y'),
        merchant=random.choice(['Amazon', 'Walmart', 'Target', 'Apple', 'Netflix', 'Uber', 'Best Buy', 'Home Depot']),
        acct_last4=f"{random.randint(1000, 9999)}",
        years=random.randint(2, 20),
        months=random.randint(2, 12),
        old_rate=f"{random.uniform(2, 5):.1f}",
        new_rate=f"{random.uniform(5, 15):.1f}",
        ref_num=f"REF{random.randint(100000, 999999)}",
        error_code=random.randint(1000, 9999),
        phone_model=random.choice(['13', '14', '15', '15 Pro']),
        ios_version=random.choice(['16.5', '17.0', '17.1', '17.2']),
        dest_acct=f"{random.randint(1000, 9999)}",
        recipient=random.choice(['John', 'Sarah', 'Mike', 'Lisa', 'Dave']),
        phone_last4=f"{random.randint(1000, 9999)}",
        num_txns=random.randint(3, 15),
        location=random.choice(['Lagos Nigeria', 'Moscow Russia', 'Beijing China', 'Mumbai India', 'Unknown']),
        home_city=random.choice(['New York', 'Chicago', 'Los Angeles', 'Houston', 'Phoenix']),
        card_last4=f"{random.randint(1000, 9999)}",
        total=f"{random.uniform(1000, 25000):.2f}",
        account_type=random.choice(['checking', 'savings', 'money market']),
        competitor_bank=random.choice(['Bank of America', 'Wells Fargo', 'Citi', 'US Bank']),
        name=f"{random.choice(first_names)} {random.choice(last_names)}",
        attorney_name=f"{random.choice(first_names)} {random.choice(last_names)}, Esq.",
        loan_type=random.choice(['mortgage', 'auto', 'personal', 'home equity']),
        credit_score=random.randint(620, 850),
        term=random.choice([5, 10, 15, 20, 30]),
        income=f"{random.randint(50000, 500000):,}",
        day=random.randint(1, 28),
        return_date=(datetime.now() - timedelta(days=random.randint(10, 45))).strftime('%m/%d/%Y'),
        tracking=f"{random.randint(10000000, 99999999)}",
        num_times=random.randint(3, 10),
        cancel_date=(datetime.now() - timedelta(days=random.randint(30, 180))).strftime('%m/%d/%Y'),
        month_num=random.randint(2, 8),
        correct_amount=f"{random.uniform(5, 500):.2f}",
        difference=f"{random.uniform(5, 200):.2f}"
    )
    
    created_at = datetime(2024, 1, 1) + timedelta(
        days=random.randint(0, 540),
        hours=random.randint(0, 23),
        minutes=random.randint(0, 59)
    )
    
    subjects = {
        'BILLING': ['Unauthorized charge', 'Fee dispute', 'Rate change inquiry', 'Double charge', 'Payment not reflected'],
        'TECHNICAL': ['Login issue', 'App crash', 'Transfer error', 'Zelle not working', '2FA problems'],
        'FRAUD': ['Unauthorized transactions', 'Identity theft', 'Wire fraud', 'ATM fraud', 'Phishing attack'],
        'ACCOUNT_CLOSURE': ['Close account request', 'Transfer to competitor', 'Deceased account holder', 'Close credit card', 'Joint account closure'],
        'GENERAL_INQUIRY': ['Loan rate inquiry', 'Account information', 'Product comparison', 'Mortgage pre-approval', 'Auto transfer setup'],
        'DISPUTE': ['Charge dispute - return', 'Service not provided', 'Subscription cancellation', 'Amount discrepancy', 'Wire recovery request']
    }
    
    support_tickets_data.append({
        'TICKET_ID': f"TKT{i+1:08d}",
        'CUSTOMER_ID': random.choice(customer_ids),
        'CREATED_AT': created_at.strftime('%Y-%m-%d %H:%M:%S'),
        'SUBJECT': random.choice(subjects[category]),
        'DESCRIPTION': description,
        'CATEGORY': category,
        'PRIORITY': random.choices(priorities, weights=priority_weights, k=1)[0],
        'CHANNEL': random.choices(ticket_channels, weights=ticket_channel_weights, k=1)[0],
        'STATUS': random.choices(ticket_statuses, weights=ticket_status_weights, k=1)[0],
        'ASSIGNED_AGENT': f"AGENT_{random.randint(1, 50):03d}",
        'RESOLUTION_TIME_HOURS': round(random.uniform(0.5, 168), 1) if random.random() > 0.3 else None,
        'SATISFACTION_SCORE': random.randint(1, 5) if random.random() > 0.4 else None
    })

support_tickets_df = pd.DataFrame(support_tickets_data)
print(f"Generated {len(support_tickets_df)} support ticket records")
print(f"Average description length: {support_tickets_df['DESCRIPTION'].str.len().mean():.0f} characters")
support_tickets_df[['TICKET_ID', 'CATEGORY', 'SUBJECT', 'DESCRIPTION']].head(2)

## Step 8: Write Data to Snowflake Temporary Tables & Unload to Stage as Parquet

We'll write the DataFrames to temporary tables, then use `COPY INTO @stage` to create Parquet files that we'll reload in Notebook 02 (demonstrating the full ingestion lifecycle).

In [ ]:
# =============================================================================
# WRITE TO SNOWFLAKE & UNLOAD TO PARQUET STAGE
# =============================================================================

# Write each DataFrame to a temporary Snowflake table
datasets = {
    'CUSTOMERS': customers_df,
    'ACCOUNTS': accounts_df,
    'TRANSACTIONS': transactions_df,
    'LOANS': loans_df,
    'CREDIT_CARDS': credit_cards_df,
    'SUPPORT_TICKETS': support_tickets_df
}

for table_name, df in datasets.items():
    print(f"Writing {table_name} ({len(df)} rows) to Snowflake...")
    snow_df = session.create_dataframe(df)
    snow_df.write.mode("overwrite").save_as_table(f"HOL_STAGING.STG_{table_name}")
    print(f"  ✓ Written to HOL_STAGING.STG_{table_name}")

print("\nAll datasets written to staging tables!")

In [ ]:
# =============================================================================
# UNLOAD TO PARQUET (using COPY INTO @stage)
# =============================================================================

tables_to_unload = ['CUSTOMERS', 'ACCOUNTS', 'TRANSACTIONS', 'LOANS', 'CREDIT_CARDS', 'SUPPORT_TICKETS']

for table_name in tables_to_unload:
    print(f"Unloading {table_name} to Parquet...")
    session.sql(f"""
        COPY INTO @HOL_LAB.HOL_PARQUET_STAGE/{table_name.lower()}/
        FROM HOL_STAGING.STG_{table_name}
        FILE_FORMAT = (TYPE = 'PARQUET')
        OVERWRITE = TRUE
        HEADER = TRUE
        MAX_FILE_SIZE = 67108864
    """).collect()
    print(f"  ✓ Unloaded to @HOL_PARQUET_STAGE/{table_name.lower()}/")

print("\n✅ All datasets unloaded to Parquet stage!")

## Step 9: Verify Stage Contents

In [ ]:
-- Verify files in stage
LIST @HOL_LAB.HOL_PARQUET_STAGE;

## Summary

Data generated and staged:

| Dataset | Rows | Stage Path |
|---------|------|------------|
| CUSTOMERS | 100K | `@HOL_PARQUET_STAGE/customers/` |
| ACCOUNTS | 200K | `@HOL_PARQUET_STAGE/accounts/` |
| TRANSACTIONS | 500K | `@HOL_PARQUET_STAGE/transactions/` |
| LOANS | 50K | `@HOL_PARQUET_STAGE/loans/` |
| CREDIT_CARDS | 100K | `@HOL_PARQUET_STAGE/credit_cards/` |
| SUPPORT_TICKETS | 50K | `@HOL_PARQUET_STAGE/support_tickets/` |

**Total:** ~1M rows of realistic consumer banking data in Parquet format, ready for ingestion.

---

**Next:** Proceed to `02_Data_Ingestion.ipynb` to load this data using COPY INTO, schema detection, and Snowpipe patterns.